In [ ]:
# apktool.bat d D:\com.jvstudios.gpstracker-254.apk

数据整理：
下载的文件先解压，然后读取每一个文件，然后看该文件里面有没有同名的apk文件，得有两个才对，如果有1个，就去别的目录同名文件里面找另一个同名apk，挪到一块去，如果有0个，则去别的目录同名文件里面找另2个同名apk，挪到一块去。

数据清洗：
遍历每一个文件夹，然后文件夹里的文件，找到同名的apk

In [1]:
import os
import re
import pandas as pd
import subprocess
import time
from pathlib import Path

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [29]:
def extract_apk_info(root_dir):
    count = 0
    all_file = set()
    rs = set()
    non_rs = set()
    # 遍历根目录及其所有子目录
    for root, dirs, files in os.walk(root_dir):
        # 获取当前文件夹名作为 apk_name (例如 ai.wizely.android)
        apk_name = os.path.basename(root)
        all_file.add(apk_name)
        # print(apk_name)
        
        # 忽略根目录本身，只处理有包名的子目录
        if not apk_name or apk_name == os.path.basename(root_dir):
            continue
            
        for file in files:
            # 检查是否为 apk 文件，并且文件名包含 apk_name（高度重合）
            if file.endswith('.apk') and apk_name in file:
                # print(file)
                rs.add(apk_name)
                count = count +1
    non_rs = all_file - rs 

    print(count)
    # print(non_rs)
    return non_rs

In [30]:
SRC_DIR = Path("1122apk/raw/1071_apk_unzip") #Path("two_examples/raw") #Path("1122apk/raw/drive-download-20260305T173717Z-1-001")  # 存放原始 APK 的文件夹
non_rs = extract_apk_info(SRC_DIR)

2142


In [23]:
len(non_rs), non_rs

(1, {'1071_apk_unzip_1'})

子目录的名字就是apk name，然后每个子目录下面应该有2个同名的apk文件，如果没有的话，就把该apkname记录下来

In [26]:
import os

def extract_apk_info(root_dir):
    invalid_apk_names = []
    valid_apk_names = []
    total_apk_dirs = 0

    # 只遍历 root_dir 下的一级子目录
    for apk_name in os.listdir(root_dir):
        apk_dir = os.path.join(root_dir, apk_name)
        # print(apk_dir)
        # 只处理子目录
        if not os.path.isdir(apk_dir):
            continue

        total_apk_dirs += 1

        # 当前子目录下的所有 .apk 文件，且文件名以 apk_name 开头
        matched_apks = [
            f for f in os.listdir(apk_dir)
            if os.path.isfile(os.path.join(apk_dir, f))
            and f.endswith(".apk")
            and f.startswith(apk_name)
        ]

        if len(matched_apks) == 2:
            valid_apk_names.append(apk_name)
        else:
            invalid_apk_names.append(apk_name)
            print(f"[INVALID] {apk_name}: found {len(matched_apks)} apk(s) -> {matched_apks}")

    print(f"总子目录数: {total_apk_dirs}")
    print(f"满足条件(正好2个同名apk)的目录数: {len(valid_apk_names)}")
    print(f"不满足条件的目录数: {len(invalid_apk_names)}")

    return invalid_apk_names

In [27]:
root_dir = r"1122apk/raw/1071_apk_unzip"
invalid_apk_names = extract_apk_info(root_dir)

[INVALID] com.iz.games.preschool.game.kindergarten.baby.children.kids.learning.educational.learn.toddler.puzzles.stories.coloring: found 0 apk(s) -> []
[INVALID] com.iz.unicorn.coloring.book.games.glitter.free.colouring.pages.pony.princess.animal.drawing.girls.color: found 0 apk(s) -> []
[INVALID] com.lonelycatgames.Xplore: found 1 apk(s) -> ['com.lonelycatgames.Xplore-44006.apk']
[INVALID] com.sriandroid.justkannada: found 1 apk(s) -> ['com.sriandroid.justkannada-4798.apk']
[INVALID] com.voicesms.writesmsbyvoice.theburraq: found 1 apk(s) -> ['com.voicesms.writesmsbyvoice.theburraq-16.apk']
[INVALID] com.whindipanchangcalendar.iwebnapp: found 1 apk(s) -> ['com.whindipanchangcalendar.iwebnapp-91.apk']
[INVALID] document.scannerapp.docscannerapp.android.imagescanner.free.camscanner.documentscanner.pdfscanner.textscanner.ocr: found 0 apk(s) -> []
[INVALID] face.makeup.beauty.photoeditor: found 1 apk(s) -> ['face.makeup.beauty.photoeditor-24.apk']
[INVALID] hairstyles.hairstylesstepbystep.

In [28]:
invalid_apk_names

['com.iz.games.preschool.game.kindergarten.baby.children.kids.learning.educational.learn.toddler.puzzles.stories.coloring',
 'com.iz.unicorn.coloring.book.games.glitter.free.colouring.pages.pony.princess.animal.drawing.girls.color',
 'com.lonelycatgames.Xplore',
 'com.sriandroid.justkannada',
 'com.voicesms.writesmsbyvoice.theburraq',
 'com.whindipanchangcalendar.iwebnapp',
 'document.scannerapp.docscannerapp.android.imagescanner.free.camscanner.documentscanner.pdfscanner.textscanner.ocr',
 'face.makeup.beauty.photoeditor',
 'hairstyles.hairstylesstepbystep.schoolhairstyles.hairstyle2.hairstylesgirls.hairstylestepbystep.hairstyle']

用apktool将指定目录里的apk解压到apk目录下

In [ ]:
# !D:/softwall_install/apktool/apktool.bat --version 2.12.1
# D:/softwall_install/apktool/apktool.bat d D:\AA_project\AA_geo_app_regulation\dataset\1122apk\raw\1071_apk_unzip\AA2_second_time\AA_first_100_batch\com.mhbl.sastasundar\com.mhbl.sastasundar-175.apk -f -o D:\AA_project\AA_geo_app_regulation\dataset\1122apk\dicompile1122apk\AA2_second_batch\AA1_first_100_batch
# -s 表示 skip sources，不反编译代码，速度提升 10 倍以上


^C


2.12.1
Press any key to continue . . . 


d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


不依赖进程退出信号

只要输出目录稳定了就继续下一个 - 文件数量/总大小在 N 秒内不再变化” → 认为已经完成

对大 APK 会自动等更久

超时后会 kill tree，绝不残留 java.exe

In [6]:
import os
import re
import pandas as pd
import subprocess
import time
from pathlib import Path

In [7]:
def dir_snapshot(out_dir: Path):
    """返回(文件数, 总字节数)作为进度快照"""
    n = 0
    total = 0
    if not out_dir.exists():
        return 0, 0
    for p in out_dir.rglob("*"):
        if p.is_file():
            n += 1
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return n, total

def looks_done(out_dir: Path):
    """快速哨兵：manifest + apktool.yml 任意命中"""
    return (out_dir / "AndroidManifest.xml").exists() and (out_dir / "apktool.yml").exists()

def run_apktool_with_completion_guard(APKTOOL, apk_path: Path, out_dir: Path,
                                      short_guard_s=30, long_guard_s=180,
                                      stable_window_s=8, poll_s=2):
    """
    - 先给 short_guard_s：如果目录已经稳定/完成，但进程不退，则 kill 放行
    - 若未完成，再给 long_guard_s：同样逻辑
    - 返回: status 字符串  cmd = [APKTOOL, "d", str(apk_path), "-f", "-o", str(output_path)]
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = ["cmd.exe", "/c", APKTOOL, "d", str(apk_path), "-f", "-o", str(out_dir)]
    p = subprocess.Popen(
        cmd,
        stdout=None,
        stderr=None,
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP
    )

    def wait_until(limit_s):
        start = time.time()
        last_change = time.time()
        last_snap = dir_snapshot(out_dir)

        while True:
            time.sleep(poll_s)

            # 进程如果自己退出了，直接结束
            rc = p.poll()
            if rc is not None:
                return "exited", rc

            # 看输出目录是否还在变化
            snap = dir_snapshot(out_dir)
            if snap != last_snap:
                last_snap = snap
                last_change = time.time()

            # 如果满足“已完成哨兵”且目录稳定了一段时间，就认为已经完成
            if looks_done(out_dir) and (time.time() - last_change) >= stable_window_s:
                return "done_but_hanging", None

            # 超时
            if (time.time() - start) >= limit_s:
                return "timeout", None

    # 第一段：短保险
    reason, rc = wait_until(short_guard_s)
    if reason == "exited" and rc == 0:
        return "ok_fast_exit"
    if reason == "done_but_hanging":
        subprocess.run(["taskkill", "/PID", str(p.pid), "/T", "/F"], capture_output=True, text=True)
        return "ok_fast_hang_killed"
    if reason == "timeout":
        # 还没完成：进入长等待（不给它直接杀）
        pass
    elif reason == "exited":
        # rc != 0 失败也进入长等待重试一次（看你需要）
        pass

    # 第二段：长等待
    reason2, rc2 = wait_until(long_guard_s)
    if reason2 == "exited" and rc2 == 0:
        return "ok_long_exit"
    if reason2 == "done_but_hanging":
        subprocess.run(["taskkill", "/PID", str(p.pid), "/T", "/F"], capture_output=True, text=True)
        return "ok_long_hang_killed"

    # 仍未完成或失败：杀掉并返回失败
    subprocess.run(["taskkill", "/PID", str(p.pid), "/T", "/F"], capture_output=True, text=True)
    return f"{apk_path}_failed_{reason2}"

In [8]:
def parse_file_info(path):
    # 1. 获取文件名：com.mhbl.sastasundar-176.ap
    filename = os.path.basename(path)
    
    # 2. 移除后缀名（无论后缀是 .ap 还是 .txt 或 .apk）
    # splitext 只会切掉最后一个点及其后面的内容
    name_without_ext = os.path.splitext(filename)[0]
    
    # 3. 提取 apk_name 和 version
    if '-' in name_without_ext:
        # rsplit('-', 1) 确保只从最右边的连字符切开一次
        parts = name_without_ext.rsplit('-', 1)
        apk_name = parts[0]   # com.mhbl.sastasundar
        version = parts[1]    # 176
    else:
        apk_name = name_without_ext
        version = "Unknown"
        
    return apk_name, version
# apk, ver = parse_file_info("com.mhbl.sastasundar-176.apk")
# apk, ver

In [9]:
# [1122apk/raw/1071_apk_unzip/AA2_second_time/AA_first_100_batch
# 1122apk/raw/1071_apk_unzip/AA2_second_time/AA_second_100_batch
# 1122apk/raw/1071_apk_unzip/AA2_second_time/AA_third_100_batch
# 1122apk/raw/1071_apk_unzip/AA2_second_time/AA_fourth_50_batch
# ]

# [
#     1122apk/dicompile1122apk/AA2_second_batch/AA1_first_100_batch
#     1122apk/dicompile1122apk/AA2_second_batch/AA3_third_100_batch
#     1122apk/dicompile1122apk/AA2_second_batch/AA5_fifth_100_batch
#     1122apk/dicompile1122apk/AA2_second_batch/AA7_seventh_100_batch
# ]

### AA3_third_time
# [1122apk/raw/1071_apk_unzip/AA3_third_time/AA_first_100_batch
# 1122apk/raw/1071_apk_unzip/AA3_third_time/AA_second_100_batch
# 1122apk/raw/1071_apk_unzip/AA3_third_time/AA_third_100_batch
# 1122apk/raw/1071_apk_unzip/AA3_third_time/AA_fourth_50_batch
# ]

# [
#     1122apk/dicompile1122apk/AA3_third_batch/AA1_first_100_batch
#     1122apk/dicompile1122apk/AA3_third_batch/AA3_third_100_batch
#     1122apk/dicompile1122apk/AA3_third_batch/AA5_fifth_100_batch
#     1122apk/dicompile1122apk/AA3_third_batch/AA7_seventh_100_batch
# ]

### AA4_forth_time
# 1122apk/raw/1071_apk_unzip/AA4_fourth_time

# 1122apk/dicompile1122apk/AA4_fourth_batch

In [11]:
# 1. 定义工具路径和目录
# 建议写法
APKTOOL = r"D:\softwall_install\apktool\apktool.bat"
SRC_DIR = Path("1122apk\\raw\\1071_apk_unzip\\AA4_fourth_time")#Path("1122apk/raw/1071_apk_unzip") #Path("1122apk/raw/50apk/4_leftover_app") #Path("1122apk/raw/drive-download-20260305T173717Z-1-001")  # 存放原始 APK 的文件夹 Path("1122apk/raw/50apk/4_leftover_app") Path("two_examples/raw")
DEST_DIR = Path("1122apk/1122apk_privacy_policy_url/AA4_fourth_batch") #Path("1122apk/dicompile1122apk")#Path("1122apk/dicompile1122apk") #Path("1122apk/dicompile1122apk") # Path("1122apk/test/")      # 解压后的输出文件夹 Path("two_examples/dicompile")

# 如果输出目录不存在则创建
DEST_DIR.mkdir(parents=True, exist_ok=True)
count = 1
results_container = []

for root, dirs, files in os.walk(SRC_DIR):
    root_path = Path(root)
    # print("root_path", root_path)
    # 获取当前文件夹名作为 apk_name (例如 ai.wizely.android)
    apk_name = os.path.basename(root)
    # print(apk_name)

    
    # 忽略根目录本身，只处理有包名的子目录
    if not apk_name or apk_name == os.path.basename(SRC_DIR):
        continue
        
    # if apk_name == "amazon.shop.barcode.scanner":
    for file in files:
        # 检查是否为 apk 文件，并且文件名包含 apk_name（高度重合）
        if file.endswith('.apk') and apk_name in file:
            # if count == 2:
                # break
            print(f"找到 {file} 文件，准备开始...")
            # # 3. 循环调用 apktool 解压
            apk_path = root_path / file
            # print('apk_path', apk_path)

            
            # 设定解压后的文件夹名
            output_path = DEST_DIR / Path(file).stem
            # 如果输出目录不存在则创建
            output_path.mkdir(parents=True, exist_ok=True)

            print(f"\n==> 开始: {file}")
            print(f"    APK : {apk_path}")
            print(f"    OUT : {output_path}")
            

            status = run_apktool_with_completion_guard(APKTOOL, apk_path, output_path,
                                        short_guard_s=30,
                                        long_guard_s=200,
                                        stable_window_s=8)
            print(apk_path.name, status)

            apk, version = parse_file_info(file)

            row = {
                "apk_name": apk_name,
                "version": version,
                "status": status # 如果没有则为 None (JSON 的 null)
            }
            results_container.append(row)

            count = count + 1

# 4. 一键转为 DataFrame
df = pd.DataFrame(results_container)

# 5. 保存结果
df.to_csv(f"{DEST_DIR}/dicompile_apks_batch_2", index=False, encoding='utf-8-sig')

print("\n--- 所有任务处理完毕 ---")

找到 thug.life.photo.sticker.maker-588.apk 文件，准备开始...

==> 开始: thug.life.photo.sticker.maker-588.apk
    APK : 1122apk\raw\1071_apk_unzip\AA4_fourth_time\thug.life.photo.sticker.maker\thug.life.photo.sticker.maker-588.apk
    OUT : 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch\thug.life.photo.sticker.maker-588
thug.life.photo.sticker.maker-588.apk ok_fast_hang_killed
找到 thug.life.photo.sticker.maker-589.apk 文件，准备开始...

==> 开始: thug.life.photo.sticker.maker-589.apk
    APK : 1122apk\raw\1071_apk_unzip\AA4_fourth_time\thug.life.photo.sticker.maker\thug.life.photo.sticker.maker-589.apk
    OUT : 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch\thug.life.photo.sticker.maker-589
thug.life.photo.sticker.maker-589.apk ok_fast_hang_killed
找到 tia.mria-23.apk 文件，准备开始...

==> 开始: tia.mria-23.apk
    APK : 1122apk\raw\1071_apk_unzip\AA4_fourth_time\tia.mria\tia.mria-23.apk
    OUT : 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch\tia.mria-23
tia.mria-23.apk ok_fast_hang_killed
找到 tia.mr

apktool 实际上已经把文件都解出来了，但进程就是不退出

In [ ]:
# ===apktool 实际上已经把文件都解出来了，但进程就是不退出，而且没有退出信号===

# # 获取所有的文件夹
# folders = [f for f in os.listdir(SRC_DIR) if os.path.isdir(os.path.join(SRC_DIR, f))]

# print(folders)



# for f in folders:
#     if f == "amazon.shop.barcode.scanner":
#         # print(f)
#         cur_path = Path(SRC_DIR/f)
#         print(cur_path)

        

        # 2. 获取所有 APK 文件
        # apks = list(cur_path.glob("*.apk"))
        # print(f"找到 {len(apks)} 个文件，准备开始...")

        # # 3. 循环调用 apktool 解压
        # for apk in apks:
        #     # 设定解压后的文件夹名
        #     output_path = DEST_DIR / apk.stem
            
        #     print(f"正在解压: {apk.name} -> {output_path}")
            
        #     # 构建命令：d 表示 decode，-f 表示强制覆盖，-o 指定输出路径
        #     cmd = [APKTOOL, "d", str(apk), "-f", "-o", str(output_path)]
            
        #     try:
        #         # 执行并捕获输出
        #         # result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        #         # 添加 shell=True 参数
        #         result = subprocess.run(cmd, capture_output=True, text=True, check=True, shell=True)

        #         print(f"✅ 成功: {apk.name}")
        #     except subprocess.CalledProcessError as e:
        #         print(f"❌ 失败: {apk.name}")
        #         print(f"错误详情: {e.stderr}")

        # print("\n--- 所有任务处理完毕 ---")

In [ ]:
# if apk_name == "amazon.shop.barcode.scanner":
#         for file in files:
#             # 检查是否为 apk 文件，并且文件名包含 apk_name（高度重合）
#             if file.endswith('.apk') and apk_name in file:
#                 print(f"找到 {file} 文件，准备开始...")
#                 # # 3. 循环调用 apktool 解压
#                 apk_path = root_path / file
#                 # print('apk_path', apk_path)

                
#                 # 设定解压后的文件夹名
#                 output_path = DEST_DIR / Path(file).stem
#                 # 如果输出目录不存在则创建
#                 output_path.mkdir(parents=True, exist_ok=True)


#                 print(f"\n==> 开始: {file}")
#                 print(f"    APK : {apk_path}")
#                 print(f"    OUT : {output_path}")

#                 status = run_apktool_with_completion_guard(APKTOOL, apk_path, output_path,
#                                            short_guard_s=30,
#                                            long_guard_s=200,
#                                            stable_window_s=8)
#                 print(apk_path.name, status)
                # print(f"正在解压: {apk_path} -> {output_path}")
                
                # 构建命令：d 表示 decode，-f 表示强制覆盖，-o 指定输出路径
                # cmd = [APKTOOL, "d", str(file), "-f", "-o", str(output_path)]
                # cmd = [APKTOOL, "d", str(apk_path), "-f", "-o", str(output_path)]
                # print(cmd)
                # try:
                #     # 执行并捕获输出
                #     # result = subprocess.run(cmd, capture_output=True, text=True, check=True)
                #     # 添加 shell=True 参数
                #     result = subprocess.run(cmd, text=True, check=True, shell=True)

                #     print(f"✅ 成功: {file}")
                # except subprocess.CalledProcessError as e:
                #     print(f"❌ 失败: {file}")
                #     # print(f"错误详情: {e.stderr}")
                
                # try:
                #     # ✅ 不 capture_output，这样你能看到实时进度，知道卡在哪一步
                #     subprocess.run(cmd, check=True, timeout=600, shell=True)
                #     print(f"✅ 成功: {file}")
                # except subprocess.TimeoutExpired:
                #     print(f"⏳ 超时(10min)跳过: {file}")
                # except subprocess.CalledProcessError as e:
                #     print(f"❌ apktool失败: {file}")
                #     print("returncode:", e.returncode)
                # except Exception as e:
                #     # ✅ 把 AttributeError 这种也抓住，不会中断后续 APK
                #     print(f"❌ 其它异常: {file} -> {type(e).__name__}: {e}")